# 01 — Data Understanding and Cleaning

## Objective
Understand the structure and quality of the EV charging dataset before making any changes.

We will inspect the dataset's dimensions, columns, data types, missing values, duplicate records, target variable, and timestamps. These observations will guide our cleaning decisions.


In [2]:
from pathlib import Path
import pandas as pd


In [3]:

# Locate the original CSV relative to this notebook.
raw_data_dir = Path("../data/raw")
csv_files = list(raw_data_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV found in data/raw/. Place the original dataset in that folder."
    )

# Avoid accidentally selecting the wrong file.
print("CSV files found:")
for file in csv_files:
    print(f" - {file.name}")

if len(csv_files) != 1:
    raise ValueError(
        "Expected exactly one CSV in data/raw/. "
        "Keep only the intended dataset there or select it explicitly."
    )

csv_path = csv_files[0]

# Read the original file without modifying it.
df = pd.read_csv(csv_path)

print(f"\nDataset loaded: {csv_path.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

CSV files found:
 - ChargingRecords.csv

Dataset loaded: ChargingRecords.csv
Rows: 72,856
Columns: 13


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


In [4]:
# Inspect column types and missing values.
print("COLUMN DATA TYPES")
display(df.dtypes.rename("data_type").to_frame())

print("\nMISSING VALUES")
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_summary.sort_values("missing_percent", ascending=False))

# Count exact duplicate rows; do not remove them yet.
print(f"\nExact duplicate rows: {df.duplicated().sum():,}")

# Inspect the target only if it exists in the actual data.
if "Demand" in df.columns:
    print("\nDEMAND SUMMARY")
    display(df["Demand"].describe())
else:
    print("\nWARNING: 'Demand' column not found.")

COLUMN DATA TYPES


,data_type
UserID,int64
ChargerID,int64
ChargerCompany,int64
Location,str
ChargerType,int64
StartDay,str
StartTime,str
EndDay,str
EndTime,str
StartDatetime,str



MISSING VALUES


,missing_count,missing_percent
UserID,0,0.0
ChargerID,0,0.0
ChargerCompany,0,0.0
Location,0,0.0
ChargerType,0,0.0
StartDay,0,0.0
StartTime,0,0.0
EndDay,0,0.0
EndTime,0,0.0
StartDatetime,0,0.0



Exact duplicate rows: 13

DEMAND SUMMARY


count    72856.000000
mean        17.444445
std         13.492728
min          0.010000
25%          7.530000
50%         14.100000
75%         23.200000
max         97.000000
Name: Demand, dtype: float64

## 2. Detailed Data-Quality Checks

The initial profile shows no missing values but identifies 13 exact duplicate occurrences. Before cleaning the dataset, we will investigate duplicate records, identifier cardinality, date-time validity, duration consistency, and invalid demand values.

The goal is to distinguish genuine data-quality problems from valid observations and to avoid removing useful information.

In [5]:

# ---------------------------------------------------------
# Detailed data-quality diagnostics
# We are inspecting the data here, not modifying it.
# ---------------------------------------------------------

# 1. Inspect exact duplicate records.
duplicate_rows = df.loc[df.duplicated(keep=False)]

print("Rows belonging to exact-duplicate groups:",
      len(duplicate_rows))
display(duplicate_rows.head(20))


# 2. Check identifier/category cardinality.
# Cardinality means the number of distinct values in a column.
candidate_id_cols = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "ChargerType",
    "Location",
]

cardinality = pd.DataFrame({
    "unique_values": df[candidate_id_cols].nunique(),
    "dtype": df[candidate_id_cols].dtypes.astype(str),
})

display(cardinality)

# Inspect examples to understand how day and time are represented.
for col in ["StartDay", "StartTime", "EndDay", "EndTime"]:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].drop_duplicates().head(8).tolist())


# 3. Parse datetime columns without changing the original columns.
# errors="coerce" converts unparseable values into NaT,
# making conversion failures easy to count.
parsed_dates = {}

for col in ["StartDatetime", "EndDatetime"]:
    parsed_dates[col] = pd.to_datetime(
        df[col],
        errors="coerce",
    )

datetime_summary = []

for col, parsed in parsed_dates.items():
    datetime_summary.append({
        "column": col,
        "valid_parsed_values": int(parsed.notna().sum()),
        "parse_failures": int(parsed.isna().sum()),
        "earliest": parsed.min(),
        "latest": parsed.max(),
    })

display(pd.DataFrame(datetime_summary))


# 4. Check duration and timestamp consistency.
# We calculate elapsed minutes from timestamps only for diagnosis.
start_dt = parsed_dates["StartDatetime"]
end_dt = parsed_dates["EndDatetime"]

valid_timestamps = start_dt.notna() & end_dt.notna()
elapsed_minutes = (
    end_dt - start_dt
).dt.total_seconds() / 60

print(
    "\nEnd timestamp earlier than start timestamp:",
    int((valid_timestamps & (end_dt < start_dt)).sum()),
)

print("Duration minimum:", df["Duration"].min())
print("Duration maximum:", df["Duration"].max())
print("Duration values <= 0:", int((df["Duration"] <= 0).sum()))

# Compare stored Duration with timestamp-derived elapsed minutes.
# Do not assume Duration is measured in minutes until verified.
duration_check = pd.DataFrame({
    "Duration_raw": df["Duration"],
    "elapsed_minutes": elapsed_minutes,
})

display(
    duration_check.loc[valid_timestamps]
    .sample(n=min(10, int(valid_timestamps.sum())), random_state=42)
    .sort_index()
)


# 5. Check for invalid numerical target values.
print("\nNegative Demand values:", int((df["Demand"] < 0).sum()))
print("Zero Demand values:", int((df["Demand"] == 0).sum()))
print("Infinite Demand values:", int(
    (~np.isfinite(df["Demand"])).sum()
))


Rows belonging to exact-duplicate groups: 26


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
28856,329,214,1,public area,0,2022-02-12,7:22:40,2022-02-12,15:44:27,2022-02-12 7:22,2022-02-12 15:44,502,54.34
28857,329,214,1,public area,0,2022-02-12,7:22:40,2022-02-12,15:44:27,2022-02-12 7:22,2022-02-12 15:44,502,54.34
28858,329,214,1,public area,0,2022-02-15,19:29:42,2022-02-15,23:03:11,2022-02-15 19:29,2022-02-15 23:03,213,23.68
28859,329,214,1,public area,0,2022-02-15,19:29:42,2022-02-15,23:03:11,2022-02-15 19:29,2022-02-15 23:03,213,23.68
28861,329,214,1,public area,0,2022-02-17,18:53:55,2022-02-18,3:38:13,2022-02-17 18:53,2022-02-18 3:38,524,58.06
28862,329,214,1,public area,0,2022-02-17,18:53:55,2022-02-18,3:38:13,2022-02-17 18:53,2022-02-18 3:38,524,58.06
40669,461,371,1,apartment,0,2022-02-22,16:17:01,2022-02-22,21:38:44,2022-02-22 16:17,2022-02-22 21:38,322,37.74
40670,461,371,1,apartment,0,2022-02-22,16:17:01,2022-02-22,21:38:44,2022-02-22 16:17,2022-02-22 21:38,322,37.74
51498,276,640,0,company,1,2022-02-26,9:23:02,2022-02-26,10:03:07,2022-02-26 9:23,2022-02-26 10:03,40,15.04
51499,276,640,0,company,1,2022-02-26,9:23:02,2022-02-26,10:03:07,2022-02-26 9:23,2022-02-26 10:03,40,15.04


,unique_values,dtype
UserID,2337,int64
ChargerID,2119,int64
ChargerCompany,2,int64
ChargerType,2,int64
Location,14,str



StartDay: 366 unique values
['2022-09-15', '2022-09-14', '2022-09-29', '2022-09-25', '2022-09-24', '2022-08-11', '2022-08-10', '2022-08-09']

StartTime: 43245 unique values
['20:54:02', '20:01:05', '18:54:30', '18:32:51', '19:30:15', '14:51:16', '18:28:42', '19:37:54']

EndDay: 365 unique values
['2022-09-15', '2022-09-14', '2022-09-30', '2022-09-26', '2022-09-24', '2022-08-11', '2022-08-09', '2022-08-07']

EndTime: 46710 unique values
['23:59:13', '21:31:04', '19:54:29', '0:16:42', '0:30:14', '16:01:15', '19:28:41', '3:51:56']


,column,valid_parsed_values,parse_failures,earliest,latest
0,StartDatetime,72856,0,2021-09-30 17:14:00,2022-09-30 23:11:00
1,EndDatetime,72856,0,2021-10-01 00:07:00,2022-09-30 23:52:00



End timestamp earlier than start timestamp: 17
Duration minimum: -29
Duration maximum: 1573
Duration values <= 0: 735


,Duration_raw,elapsed_minutes
4772,122,418.0
8241,184,184.0
11373,480,480.0
19236,52,53.0
38786,53,53.0
39804,260,260.0
41963,438,438.0
54135,23,24.0
64483,306,305.0
72089,327,327.0



Negative Demand values: 0
Zero Demand values: 0
Infinite Demand values: 0


### 3. Validate Timestamp and Duration Consistency

The dataset contains non-positive duration values and records where the end timestamp precedes the start timestamp. Some duration values also differ from the elapsed time calculated using the stored datetime columns.

The `StartDatetime` and `EndDatetime` columns appear to have minute-level precision, while `StartTime` and `EndTime` contain seconds. Therefore, we will reconstruct timestamps using the separate date and time columns and compare the resulting elapsed time with the stored `Duration`.

These checks are diagnostic only. We will not modify or remove any records until we understand the discrepancies.

In [6]:

# ---------------------------------------------------------
# 1. Reconstruct timestamps using the most precise fields.
# StartTime and EndTime include seconds, unlike the
# minute-level StartDatetime and EndDatetime columns.
# ---------------------------------------------------------

start_exact = pd.to_datetime(
    df["StartDay"].astype(str).str.strip()
    + " "
    + df["StartTime"].astype(str).str.strip(),
    errors="coerce",
)

end_exact = pd.to_datetime(
    df["EndDay"].astype(str).str.strip()
    + " "
    + df["EndTime"].astype(str).str.strip(),
    errors="coerce",
)

# Calculate elapsed time in minutes using exact timestamps.
elapsed_minutes = (
    (end_exact - start_exact).dt.total_seconds() / 60
)

# Parse the existing datetime columns for consistency checks.
stored_start = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)
stored_end = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)


# ---------------------------------------------------------
# 2. Check whether the stored datetime columns agree with
# the reconstructed timestamps at minute-level precision.
# ---------------------------------------------------------

start_match = stored_start.eq(start_exact.dt.floor("min"))
end_match = stored_end.eq(end_exact.dt.floor("min"))

print("EXACT TIMESTAMP VALIDATION")
print("Start timestamp parsing failures:", start_exact.isna().sum())
print("End timestamp parsing failures:", end_exact.isna().sum())

print(
    "StartDatetime matches reconstructed start:",
    f"{start_match.sum():,} / {len(df):,}",
)
print(
    "EndDatetime matches reconstructed end:",
    f"{end_match.sum():,} / {len(df):,}",
)

print(
    "End before start using reconstructed timestamps:",
    int((end_exact < start_exact).sum()),
)


# ---------------------------------------------------------
# 3. Compare stored Duration against elapsed minutes.
# We compare only records with positive durations and valid,
# non-negative elapsed times.
# ---------------------------------------------------------

duration_difference = (df["Duration"] - elapsed_minutes).abs()

comparable = (
    start_exact.notna()
    & end_exact.notna()
    & (end_exact >= start_exact)
    & df["Duration"].gt(0)
)

# A difference of at most 2 minutes is a provisional
# screening tolerance, not an automatic cleaning rule.
within_tolerance = (
    comparable & duration_difference.le(2)
)

outside_tolerance = (
    comparable & duration_difference.gt(2)
)

n_comparable = int(comparable.sum())

print("\nDURATION CONSISTENCY")
print("Comparable records:", f"{n_comparable:,}")
print(
    "Duration within 2 minutes of elapsed time:",
    f"{within_tolerance.sum():,}",
)
print(
    "Duration differs by more than 2 minutes:",
    f"{outside_tolerance.sum():,}",
)

if n_comparable:
    print(
        "Percentage within tolerance:",
        f"{within_tolerance.sum() / n_comparable * 100:.2f}%",
    )


# ---------------------------------------------------------
# 4. Create a temporary diagnostic table.
# This does not change the original df.
# ---------------------------------------------------------

diagnostics = df[
    [
        "UserID",
        "ChargerID",
        "StartDay",
        "StartTime",
        "EndDay",
        "EndTime",
        "StartDatetime",
        "EndDatetime",
        "Duration",
        "Demand",
    ]
].copy()

diagnostics["start_exact"] = start_exact
diagnostics["end_exact"] = end_exact
diagnostics["elapsed_minutes"] = elapsed_minutes
diagnostics["duration_difference"] = (
    df["Duration"] - elapsed_minutes
).abs()


# Show the records with the largest discrepancies.
print("\n10 LARGEST DURATION DISCREPANCIES")

display(
    diagnostics.loc[outside_tolerance]
    .sort_values("duration_difference", ascending=False)
    .head(10)
)


# Inspect non-positive duration records separately.
print("\nSAMPLE OF NON-POSITIVE DURATION RECORDS")

display(
    diagnostics.loc[df["Duration"].le(0)].head(15)
)


EXACT TIMESTAMP VALIDATION
Start timestamp parsing failures: 0
End timestamp parsing failures: 0
StartDatetime matches reconstructed start: 72,856 / 72,856
EndDatetime matches reconstructed end: 72,856 / 72,856
End before start using reconstructed timestamps: 17

DURATION CONSISTENCY
Comparable records: 72,121
Duration within 2 minutes of elapsed time: 69,463
Duration differs by more than 2 minutes: 2,658
Percentage within tolerance: 96.31%

10 LARGEST DURATION DISCREPANCIES


,UserID,ChargerID,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,start_exact,end_exact,elapsed_minutes,duration_difference
68973,285,1784,2022-05-14,13:59:33,2022-05-17,20:36:35,2022-05-14 13:59,2022-05-17 20:36,397,22.93,2022-05-14 13:59:33,2022-05-17 20:36:35,4717.033333,4320.033333
18795,0,114,2021-10-01,18:31:08,2021-10-04,13:17:25,2021-10-01 18:31,2021-10-04 13:17,101,2.30,2021-10-01 18:31:08,2021-10-04 13:17:25,4006.283333,3905.283333
13420,0,69,2022-03-30,22:04:34,2022-04-02,15:00:15,2022-03-30 22:04,2022-04-02 15:00,157,16.70,2022-03-30 22:04:34,2022-04-02 15:00:15,3895.683333,3738.683333
8022,0,33,2022-03-05,9:09:11,2022-03-07,12:09:13,2022-03-05 9:09,2022-03-07 12:09,132,14.40,2022-03-05 09:09:11,2022-03-07 12:09:13,3060.033333,2928.033333
72429,1860,2499,2022-03-24,22:08:20,2022-03-27,13:01:11,2022-03-24 22:08,2022-03-27 13:01,893,23.03,2022-03-24 22:08:20,2022-03-27 13:01:11,3772.850000,2879.850000
72198,1093,2414,2021-10-25,8:57:16,2021-10-27,9:00:02,2021-10-25 8:57,2021-10-27 9:00,3,15.32,2021-10-25 08:57:16,2021-10-27 09:00:02,2882.766667,2879.766667
4911,0,21,2021-10-23,23:54:59,2021-10-25,23:50:59,2021-10-23 23:54,2021-10-25 23:50,167,5.10,2021-10-23 23:54:59,2021-10-25 23:50:59,2876.000000,2709.000000
21530,0,140,2022-01-21,23:14:22,2022-01-23,22:28:59,2022-01-21 23:14,2022-01-23 22:28,214,7.20,2022-01-21 23:14:22,2022-01-23 22:28:59,2834.616667,2620.616667
21540,0,140,2021-11-13,23:05:57,2021-11-15,20:08:43,2021-11-13 23:05,2021-11-15 20:08,177,5.40,2021-11-13 23:05:57,2021-11-15 20:08:43,2702.766667,2525.766667
4862,0,21,2022-03-08,18:24:41,2022-03-10,12:49:59,2022-03-08 18:24,2022-03-10 12:49,229,8.00,2022-03-08 18:24:41,2022-03-10 12:49:59,2545.300000,2316.300000



SAMPLE OF NON-POSITIVE DURATION RECORDS


,UserID,ChargerID,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,start_exact,end_exact,elapsed_minutes,duration_difference
32,0,1,2022-06-10,18:52:53,2022-06-10,18:52:53,2022-06-10 18:52,2022-06-10 18:52,0,0.54,2022-06-10 18:52:53,2022-06-10 18:52:53,0.000000,0.000000
33,0,1,2022-06-10,19:14:26,2022-06-10,19:14:26,2022-06-10 19:14,2022-06-10 19:14,0,0.55,2022-06-10 19:14:26,2022-06-10 19:14:26,0.000000,0.000000
50,0,1,2022-03-01,12:45:35,2022-03-01,13:34:38,2022-03-01 12:45,2022-03-01 13:34,0,5.58,2022-03-01 12:45:35,2022-03-01 13:34:38,49.050000,49.050000
57,0,1,2022-03-13,9:49:06,2022-03-13,10:41:42,2022-03-13 9:49,2022-03-13 10:41,0,4.66,2022-03-13 09:49:06,2022-03-13 10:41:42,52.600000,52.600000
66,0,1,2022-02-11,8:45:14,2022-02-11,10:00:14,2022-02-11 8:45,2022-02-11 10:00,0,8.34,2022-02-11 08:45:14,2022-02-11 10:00:14,75.000000,75.000000
106,0,1,2022-06-09,14:40:16,2022-06-09,14:40:39,2022-06-09 14:40,2022-06-09 14:40,0,0.02,2022-06-09 14:40:16,2022-06-09 14:40:39,0.383333,0.383333
141,0,1,2022-08-01,11:14:23,2022-08-01,11:14:42,2022-08-01 11:14,2022-08-01 11:14,0,0.02,2022-08-01 11:14:23,2022-08-01 11:14:42,0.316667,0.316667
244,1068,1,2021-12-26,16:34:37,2021-12-26,17:01:55,2021-12-26 16:34,2021-12-26 17:01,0,3.03,2021-12-26 16:34:37,2021-12-26 17:01:55,27.300000,27.300000
246,1068,1,2021-12-27,17:29:49,2021-12-27,20:24:22,2021-12-27 17:29,2021-12-27 20:24,0,17.32,2021-12-27 17:29:49,2021-12-27 20:24:22,174.550000,174.550000
254,0,1,2022-06-26,14:58:45,2022-06-26,14:59:14,2022-06-26 14:58,2022-06-26 14:59,0,0.05,2022-06-26 14:58:45,2022-06-26 14:59:14,0.483333,0.483333


### 4. Data Cleaning and Validation

The diagnostic analysis identified redundant duplicate records, a small number of sessions with invalid chronological ordering, and inconsistencies between recorded durations and timestamp-derived elapsed times.

The cleaning strategy is to remove exact duplicate occurrences and records with invalid timestamp ordering, while preserving the original dataset. We will reconstruct precise datetime columns and create a separate `DurationValidated` field.

A duration is considered valid for this project if it is positive and differs from timestamp-derived elapsed time by no more than two minutes. This is a documented screening rule, not a universal industry standard.

Records with invalid durations will be retained when their other information remains useful. The original `Duration` column will remain available for traceability.


In [7]:


# Keep the original DataFrame unchanged.
rows_original = len(df)

# ---------------------------------------------------------
# 1. Remove exact duplicate occurrences.
# Keep one copy of each fully identical record.
# ---------------------------------------------------------
duplicate_count = int(df.duplicated().sum())

df_clean = df.drop_duplicates().copy()

rows_after_dedup = len(df_clean)


# ---------------------------------------------------------
# 2. Reconstruct precise timestamps.
# The separate day/time columns contain seconds, whereas
# the original datetime columns contain minute-level times.
# ---------------------------------------------------------
df_clean["StartDatetime"] = pd.to_datetime(
    df_clean["StartDay"].astype(str).str.strip()
    + " "
    + df_clean["StartTime"].astype(str).str.strip(),
    errors="coerce",
)

df_clean["EndDatetime"] = pd.to_datetime(
    df_clean["EndDay"].astype(str).str.strip()
    + " "
    + df_clean["EndTime"].astype(str).str.strip(),
    errors="coerce",
)


# ---------------------------------------------------------
# 3. Identify records without valid timestamps.
# A valid start timestamp is essential for our time features.
# We also need an end timestamp to validate duration.
# ---------------------------------------------------------
invalid_timestamp_mask = (
    df_clean["StartDatetime"].isna()
    | df_clean["EndDatetime"].isna()
)

invalid_timestamp_count = int(invalid_timestamp_mask.sum())

df_clean = df_clean.loc[~invalid_timestamp_mask].copy()


# ---------------------------------------------------------
# 4. Remove records whose end timestamp precedes the start.
# These records have invalid chronological ordering.
# ---------------------------------------------------------
invalid_order_mask = (
    df_clean["EndDatetime"] < df_clean["StartDatetime"]
)

invalid_order_count = int(invalid_order_mask.sum())

df_clean = df_clean.loc[~invalid_order_mask].copy()


# ---------------------------------------------------------
# 5. Validate Duration without changing the original values.
# First calculate elapsed minutes from precise timestamps.
# ---------------------------------------------------------
elapsed_minutes = (
    df_clean["EndDatetime"] - df_clean["StartDatetime"]
).dt.total_seconds() / 60

duration_difference = (
    df_clean["Duration"] - elapsed_minutes
).abs()

# Duration is validated only if it is positive and its
# difference from elapsed time is within our 2-minute rule.
duration_valid_mask = (
    df_clean["Duration"].gt(0)
    & elapsed_minutes.ge(0)
    & duration_difference.le(2)
)

# Preserve Duration as supplied. Store the validated value
# separately; invalid values become missing in this new field.
df_clean["DurationValidated"] = df_clean["Duration"].where(
    duration_valid_mask
)

# Keep a reason for the duration-validation outcome.
df_clean["DurationValidationStatus"] = np.select(
    [
        df_clean["Duration"].le(0),
        duration_difference.gt(2),
    ],
    [
        "non_positive",
        "mismatch_over_2min",
    ],
    default="valid",
)


# ---------------------------------------------------------
# 6. Verify the target before saving the cleaned dataset.
# We should never silently remove or alter the target.
# ---------------------------------------------------------
demand_is_valid = (
    df_clean["Demand"].notna().all()
    and np.isfinite(df_clean["Demand"].to_numpy()).all()
    and df_clean["Demand"].ge(0).all()
)

if not demand_is_valid:
    raise ValueError(
        "Demand contains missing, infinite, or negative values. "
        "Investigate these records before proceeding."
    )


# ---------------------------------------------------------
# 7. Save the cleaned dataset to data/processed/.
# Path assumes this notebook runs from the notebooks folder.
# ---------------------------------------------------------
output_path = Path("../data/processed/ev_charging_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)


# ---------------------------------------------------------
# 8. Print a concise cleaning report.
# ---------------------------------------------------------
print("DATA CLEANING REPORT")
print("-" * 40)
print(f"Original rows: {rows_original:,}")
print(f"Duplicate occurrences removed: {duplicate_count:,}")
print(f"Rows after deduplication: {rows_after_dedup:,}")
print(f"Rows with invalid timestamps removed: {invalid_timestamp_count:,}")
print(f"Rows with end before start removed: {invalid_order_count:,}")
print(f"Final cleaned rows: {len(df_clean):,}")
print(f"Final columns: {df_clean.shape[1]:,}")

print("\nDURATION VALIDATION")
print(
    df_clean["DurationValidationStatus"]
    .value_counts()
    .reindex(
        ["valid", "non_positive", "mismatch_over_2min"],
        fill_value=0,
    )
    .to_string()
)

print(
    "\nValidated duration values:",
    f"{df_clean['DurationValidated'].notna().sum():,}",
)

print(
    "Rows without a validated duration:",
    f"{df_clean['DurationValidated'].isna().sum():,}",
)

print("\nDEMAND VALIDATION")
print("All retained Demand values are finite and non-negative.")

print(f"\nSaved cleaned dataset to: {output_path}")


DATA CLEANING REPORT
----------------------------------------
Original rows: 72,856
Duplicate occurrences removed: 13
Rows after deduplication: 72,843
Rows with invalid timestamps removed: 0
Rows with end before start removed: 17
Final cleaned rows: 72,826
Final columns: 15

DURATION VALIDATION
DurationValidationStatus
valid                 69450
non_positive            718
mismatch_over_2min     2658

Validated duration values: 69,450
Rows without a validated duration: 3,376

DEMAND VALIDATION
All retained Demand values are finite and non-negative.

Saved cleaned dataset to: ..\data\processed\ev_charging_cleaned.csv


In [8]:
# Use only records whose duration passed our validation rule.
duration_analysis = df_clean.dropna(
    subset=["DurationValidated", "Demand"]
).copy()

print("Records used for duration-demand analysis:",
      len(duration_analysis))

# Pearson correlation measures the strength and direction
# of a linear relationship between duration and demand.
correlation = duration_analysis[
    ["DurationValidated", "Demand"]
].corr(method="pearson").iloc[0, 1]

print(f"Duration-Demand Pearson correlation: {correlation:.3f}")

Records used for duration-demand analysis: 69450
Duration-Demand Pearson correlation: 0.645


# 5. Summary, Findings, and Conclusion

## 5.1 Objective Achieved

The objective of this notebook was to understand the EV charging dataset, investigate its data quality, validate the timestamp and duration fields, and prepare a cleaned dataset for subsequent analysis and machine learning.

The cleaning process was completed without modifying the original source CSV. The resulting dataset was saved separately for reproducibility.

## 5.2 Dataset Overview

The original dataset contained **72,856 charging records and 13 columns**.

The dataset included:
- **Identifiers:** `UserID`, `ChargerID`
- **Charging metadata:** `ChargerCompany`, `ChargerType`, `Location`
- **Temporal information:** `StartDay`, `StartTime`, `EndDay`, `EndTime`, `StartDatetime`, `EndDatetime`
- **Session measurement:** `Duration`
- **Prediction target:** `Demand`

During the initial data profiling, no missing values were detected in any of the original columns. The initial `Demand` distribution had a mean of approximately 17.44, a median of 14.10, and a standard deviation of approximately 13.49. Demand ranged from 0.01 to 97.00.

These statistics describe the initial dataset before cleaning; final statistics can differ slightly after removing duplicate and chronologically invalid records.

## 5.3 Data-Quality Findings

### A. Missing Values

The initial inspection detected zero missing values across all 13 original columns.

Consequently, no missing-value imputation was required for the original fields. This does not automatically guarantee that every value was valid, so additional checks were necessary.

### B. Duplicate Records

The initial inspection identified **13 excess exact duplicate occurrences**. Inspecting duplicate groups displayed 26 rows because both the retained copies and their repetitions were shown.

The redundant occurrences were removed using `drop_duplicates()`, retaining one copy of each fully identical record.

The row count decreased from 72,856 to 72,843 after deduplication.

### C. Datetime Validation

The original `StartDatetime` and `EndDatetime` columns were stored as strings with minute-level precision. The separate date and time columns contained seconds.

Both timestamps were successfully reconstructed from their respective date and time components. All 72,856 original records parsed successfully, and the reconstructed timestamps agreed with the stored datetime fields at minute-level precision.

After deduplication, 17 records had an end timestamp earlier than the start timestamp. These records were excluded because their chronological ordering was invalid.

No timestamp-parsing failures were observed.

### D. Duration Validation

The stored `Duration` values were compared with elapsed time calculated from the reconstructed timestamps.

The project used the following screening rule:

A duration was considered validated if it was positive and differed from timestamp-derived elapsed time by no more than two minutes.

The two-minute tolerance was a practical project assumption for identifying inconsistent records, not a universal industry standard.

The final duration-validation results were:

| Validation status | Records |
|---|---:|
| Validated durations | 69,450 |
| Non-positive durations | 718 |
| Mismatches greater than two minutes | 2,658 |
| **Total cleaned records assessed** | **72,826** |

The original `Duration` column was preserved. A new `DurationValidated` column was created to retain duration values that passed the validation rule, while other duration values were represented as missing in that new column.

A `DurationValidationStatus` column records the reason each duration passed or failed the screening rule.

Importantly, records with unreliable duration values were not automatically deleted if their other information remained usable.

### E. Demand Validation

The cleaned dataset was checked for missing, negative, and infinite values in `Demand`.

All retained demand values were finite and non-negative. The target variable was preserved for subsequent regression and classification tasks.

## 5.4 Key Analytical Finding: Duration and Demand

An initial correlation analysis using the 69,450 validated duration observations produced a Pearson correlation coefficient of **0.645** between `DurationValidated` and `Demand`.

This indicates a positive linear association in the observed dataset: sessions with longer validated durations tended to have higher demand.

However, this result does not establish causation. It also does not justify using actual session duration as an input when predicting demand at the beginning of a session, because the duration is not yet known at that time.

Duration will therefore be used for appropriately scoped exploratory analysis rather than as a predictor in the primary session-start regression and classification models.

## 5.5 Cleaning Decisions and Rationale

The following transformations were applied:

1. Removed 13 redundant duplicate occurrences.
2. Reconstructed `StartDatetime` and `EndDatetime` from the more precise date and time components.
3. Removed 17 records with invalid chronological ordering.
4. Validated durations against elapsed timestamp differences.
5. Preserved records with unreliable durations when their remaining information was still usable.
6. Retained `Demand` after confirming that all remaining values were finite and non-negative.
7. Saved the cleaned dataset separately from the original CSV.

These decisions preserve useful observations while making known data-quality limitations explicit.

## 5.6 Final Cleaning Results

| Metric | Result |
|---|---:|
| Original records | 72,856 |
| Duplicate occurrences removed | 13 |
| Records removed for invalid chronology | 17 |
| **Final cleaned records** | **72,826** |
| Original columns | 13 |
| Final columns | 15 |
| Validated duration observations | 69,450 |
| Records without a validated duration | 3,376 |
| Missing values in the original columns at initial inspection | 0 |

The final dataset contains two additional columns: `DurationValidated` and `DurationValidationStatus`. The timestamp columns also now contain reconstructed datetime values with second-level precision.

**Output file:** `data/processed/ev_charging_cleaned.csv`

## 5.7 Limitations and Precautions

- The two-minute duration-validation threshold is a project assumption and should be disclosed.
- A failed duration check does not prove that every other field in the corresponding record is incorrect.
- Correlation between duration and demand does not establish causation.
- End-of-session information must not be used as an input to a model intended to make predictions before the session begins.
- Data cleaning does not guarantee that all possible data-quality issues have been resolved; the dataset should continue to be checked during EDA and modeling.

## 5.8 Conclusion and Next Steps

The data-understanding and cleaning stage is complete. The cleaned dataset contains **72,826 records**, with redundant exact duplicates and records with invalid chronological ordering removed. Duration reliability is explicitly tracked rather than assumed.

In the next notebook, feature engineering will extract useful temporal information from `StartDatetime`, derive the High/Normal Demand classification target using the GUVI-prescribed median threshold, and establish the candidate predictor variables for regression and classification.

The cleaned dataset will remain unchanged; the engineered dataset will be saved as a separate artifact for downstream tasks.